# Trajectory Inference

Infer cell trajectories and pseudotime ordering.

This notebook:
1. Runs PAGA (Partition-based Graph Abstraction) analysis
2. Computes diffusion pseudotime if a root cell type is specified
3. Visualizes trajectories on UMAP
4. Saves trajectory results

In [ ]:
# Parameters (injected by Papermill)
input_h5ad_path: str = "/data/results/experiment/03_clustered.h5ad"
experiment_name: str = "experiment"
root_cell_type: str = ""
method: str = "paga"
bioaf_api_url: str = "http://localhost:8000"
experiment_id: str = ""

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, frameon=False, figsize=(6, 4))

results_dir = f"/data/results/{experiment_name}"
figures_dir = os.path.join(results_dir, "figures", "trajectory")
os.makedirs(figures_dir, exist_ok=True)

# Normalize root_cell_type: treat None or empty string as unset
if not root_cell_type or root_cell_type == "None":
    root_cell_type = None

## 1. Load Data

In [ ]:
adata = sc.read_h5ad(input_h5ad_path)
print(f"Loaded: {adata.n_obs} cells x {adata.n_vars} genes")

# Ensure leiden clustering is present
if "leiden" not in adata.obs.columns:
    print("No leiden clusters found, running clustering with default resolution...")
    if "neighbors" not in adata.uns:
        sc.pp.neighbors(adata)
    sc.tl.leiden(adata)

cluster_col = "leiden"
print(f"Clusters: {adata.obs[cluster_col].nunique()}")

## 2. PAGA Analysis

Compute the PAGA graph to estimate connectivity between clusters.

In [ ]:
sc.tl.paga(adata, groups=cluster_col)

fig, ax = plt.subplots(figsize=(8, 6))
sc.pl.paga(adata, threshold=0.03, ax=ax, show=False,
           title="PAGA Graph", frameon=False)
fig.savefig(os.path.join(figures_dir, "paga_graph.png"), bbox_inches="tight")
plt.show()

In [ ]:
# PAGA-initialized UMAP for smoother trajectory visualization
sc.tl.umap(adata, init_pos="paga")

fig, ax = plt.subplots(figsize=(8, 6))
sc.pl.umap(adata, color=cluster_col, legend_loc="on data", ax=ax, show=False,
           title="UMAP (PAGA-initialized)")
fig.savefig(os.path.join(figures_dir, "umap_paga_init.png"), bbox_inches="tight")
plt.show()

## 3. Diffusion Pseudotime

If a root cell type is specified, compute diffusion pseudotime (DPT)
to order cells along a trajectory.

In [ ]:
# Compute diffusion map (required for DPT)
sc.tl.diffmap(adata)
print("Diffusion map computed")

In [ ]:
if root_cell_type is not None:
    # Find root cell as the cell in the specified cluster closest to the
    # diffusion component minimum
    root_mask = adata.obs[cluster_col] == str(root_cell_type)
    if root_mask.sum() == 0:
        print(f"Warning: root_cell_type '{root_cell_type}' not found in '{cluster_col}'.")
        print(f"Available groups: {sorted(adata.obs[cluster_col].unique())}")
        print("Skipping DPT computation.")
        root_cell_type = None
    else:
        root_idx = np.where(root_mask)[0]
        dc1_vals = adata.obsm["X_diffmap"][root_idx, 0]
        adata.uns["iroot"] = root_idx[np.argmin(dc1_vals)]
        print(f"Root cell set from cluster '{root_cell_type}' (index {adata.uns['iroot']})")

        sc.tl.dpt(adata)
        print("Diffusion pseudotime computed")
else:
    print("No root_cell_type specified. Skipping DPT. Set root_cell_type to a "
          "cluster label to enable pseudotime ordering.")

In [ ]:
if "dpt_pseudotime" in adata.obs.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sc.pl.umap(adata, color="dpt_pseudotime", ax=axes[0], show=False,
               title="Diffusion Pseudotime", color_map="viridis")
    sc.pl.umap(adata, color=cluster_col, legend_loc="on data", ax=axes[1],
               show=False, title="Clusters")
    plt.tight_layout()
    fig.savefig(os.path.join(figures_dir, "pseudotime_umap.png"), bbox_inches="tight")
    plt.show()
else:
    print("DPT not computed -- no pseudotime plot generated.")

## 4. PAGA Path Visualization

In [ ]:
# PAGA overlaid on UMAP
fig, ax = plt.subplots(figsize=(8, 6))
sc.pl.paga_compare(adata, threshold=0.03, ax=ax, show=False,
                   title="PAGA on UMAP", frameon=False)
fig.savefig(os.path.join(figures_dir, "paga_on_umap.png"), bbox_inches="tight")
plt.show()

## 5. Save Results

In [ ]:
output_path = os.path.join(results_dir, "05_trajectory.h5ad")
adata.write_h5ad(output_path)

# Export pseudotime if available
if "dpt_pseudotime" in adata.obs.columns:
    pt_csv = os.path.join(results_dir, "05_pseudotime.csv")
    adata.obs[[cluster_col, "dpt_pseudotime"]].to_csv(pt_csv)
    print(f"Saved pseudotime table to: {pt_csv}")

# Export PAGA connectivity
paga_csv = os.path.join(results_dir, "05_paga_connectivities.csv")
paga_conn = pd.DataFrame(
    adata.uns["paga"]["connectivities"].toarray(),
    index=adata.obs[cluster_col].cat.categories,
    columns=adata.obs[cluster_col].cat.categories,
)
paga_conn.to_csv(paga_csv)

print(f"Saved trajectory data to: {output_path}")
print(f"Saved PAGA connectivities to: {paga_csv}")
print(f"Method: {method}")